### Install and Import Libraries

In [ ]:
!pip install
!pip install mlflow

import pandas as pd
import time
import optuna
import os
import mlflow
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, classification_report
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

ERROR: You must give at least one requirement to install (see "pip help install")
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 5.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.2/44.2 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 118.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 125.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 103.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.4/228.4 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.5/136.5 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

### Load Data

In [ ]:
df = pd.read_csv("/content/WA_Fn-UseC_-Telco-Customer-Churn.csv")
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [ ]:
CATEGORICAL_FEATURES = ['gender', 'SeniorCitizen', 'Partner',
                        'Dependents', 'PhoneService', 'MultipleLines',
                        'InternetService', 'OnlineSecurity', 'OnlineBackup',
                        'DeviceProtection', 'TechSupport', 'StreamingTV',
                        'StreamingMovies', 'Contract', 'PaperlessBilling',
                        'PaymentMethod']
NUMERICAL_FEATURES = ['tenure', 'MonthlyCharges', 'TotalCharges']

In [ ]:
for category in CATEGORICAL_FEATURES:
  print(f"{category}: {df[category].unique()}")

gender: ['Female' 'Male']
SeniorCitizen: [0 1]
Partner: ['Yes' 'No']
Dependents: ['No' 'Yes']
PhoneService: ['No' 'Yes']
MultipleLines: ['No phone service' 'No' 'Yes']
InternetService: ['DSL' 'Fiber optic' 'No']
OnlineSecurity: ['No' 'Yes' 'No internet service']
OnlineBackup: ['Yes' 'No' 'No internet service']
DeviceProtection: ['No' 'Yes' 'No internet service']
TechSupport: ['No' 'Yes' 'No internet service']
StreamingTV: ['No' 'Yes' 'No internet service']
StreamingMovies: ['No' 'Yes' 'No internet service']
Contract: ['Month-to-month' 'One year' 'Two year']
PaperlessBilling: ['Yes' 'No']
PaymentMethod: ['Electronic check' 'Mailed check' 'Bank transfer (automatic)'
 'Credit card (automatic)']


In [ ]:
df.describe()

,SeniorCitizen,tenure,MonthlyCharges
count,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.761692
std,0.368612,24.559481,30.090047
min,0.000000,0.000000,18.250000
25%,0.000000,9.000000,35.500000
50%,0.000000,29.000000,70.350000
75%,0.000000,55.000000,89.850000
max,1.000000,72.000000,118.750000


In [ ]:
df['Churn'].value_counts()

,count
Churn,
No,5174
Yes,1869


### Data Preprocessing

i. Drop columns not relevant to learning

ii. Split dataset into training and test sets, and into inputs and labels

iii. Fill/make provision to fill NA values

iv. Normalise/scale numerical features

v. Encode categorical features. We treat all the categorical features as nominal, as none has a strict order or hierarchy.

In [ ]:
# Drop the customerID column
df = df.drop('customerID', axis=1)

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

In [ ]:
# Create training and test sets
X = df.drop('Churn', axis=1)
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=42, stratify=y)

In [ ]:
# Create validation set
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.15, random_state=42, stratify=y_train)

In [ ]:
# Scale numerical features and fill null values
numerical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

In [ ]:
# Encode categorical features and fill null values
categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

In [ ]:
# Create processing pipeline
processor = ColumnTransformer([
    ('numerical', numerical_pipeline, NUMERICAL_FEATURES),
    ('categorical', categorical_pipeline, CATEGORICAL_FEATURES)
])

### Data Transformation

In [ ]:
X_train = processor.fit_transform(X_train)
X_val = processor.transform(X_val)
X_test = processor.transform(X_test)

In [ ]:
X_train.shape, X_val.shape, X_test.shape

((5088, 46), (898, 46), (1057, 46))

In [ ]:
TARGET_MAPPING = {
    "No": 0,
    "Yes": 1
}
y_train = y_train.map(TARGET_MAPPING)
y_val = y_val.map(TARGET_MAPPING)
y_test = y_test.map(TARGET_MAPPING)

In [ ]:
y_train[:5]

,Churn
3001,0
2865,0
2715,0
6799,0
6788,0


### Model Training

Criteria for choosing models to try out before choosing a final model.

i. It's a classification problem, so we'd train suitable for classification

ii. As a base model, we'd train a Logistic Regression model.

iii. For this problem, interpretability isn't really needed. So we'd train and compare performances of Boosted Tree algorithms - Random Forest, XGBoost and LightGBM

Considering the business context, Churn Prediction is asymmetric in cost. False Negatives (missed churners) cost more than False Positives (loyal customers predicted as churners).
False Positives still has cost, however, so we would not simply optimise for a high recall.

We assume that the business cost of missing a churner is 6 times more than that of targeting a loyal customer.
So for our evaluation metric, we would define a function that models the business cost.

In [ ]:
# When consider which model to choose after training a few options, consider your preferred metrics as well as training and inference time.

Evaluation Metric

In [ ]:
def business_cost(y_true, y_pred, precision_threshold=0.45, penalty_severity=5000):
    # Confusion matrix components
    tp = sum((y_true == 1) & (y_pred == 1))
    fp = sum((y_true == 0) & (y_pred == 1))
    fn = sum((y_true == 1) & (y_pred == 0))

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0

    # Base cost (6.0x for FN, 1.0x for FP)
    base_cost = (6.0 * fn) + (1.0 * fp)

    # Bottleneck penalty if precision falls below threshold
    penalty = max(0.0, precision_threshold - precision) * penalty_severity

    return base_cost + penalty

Baseline Model

In [ ]:
# Logistic Regression
logistic_reg = LogisticRegression(random_state=42)

start_train = time.time()
logistic_reg.fit(X_train, y_train)
print(f"Training time: {(time.time() - start_train):.2f} seconds")

Training time: 0.05 seconds


In [ ]:
start_pred = time.time()
y_pred = logistic_reg.predict(X_val)
print(f"Prediction time: {(time.time() - start_pred):.2f} seconds")
print(f"Cost: {business_cost(y_val, y_pred)}\n")
print(classification_report(y_val, y_pred))

Prediction time: 0.00 seconds
Cost: 753.0

              precision    recall  f1-score   support

           0       0.84      0.90      0.87       660
           1       0.64      0.52      0.58       238

    accuracy                           0.80       898
   macro avg       0.74      0.71      0.72       898
weighted avg       0.79      0.80      0.79       898



In [ ]:
# Reduce prediction threshold for a higher recall (to capture more churners)
for threshold in [0.15, 0.2, 0.25, 0.3, 0.35, 0.4]:
  y_pred = logistic_reg.predict_proba(X_val)[:,1] > threshold
  print(f"Threshold: {threshold}")
  print(f"Precision: {precision_score(y_val, y_pred, pos_label=1)}")
  print(f"Recall: {recall_score(y_val, y_pred, pos_label=1)}")
  print(f"Cost: {business_cost(y_val, y_pred)}")
  print("\n")

Threshold: 0.15
Precision: 0.433264887063655
Recall: 0.8865546218487395
Cost: 521.675564681725


Threshold: 0.2
Precision: 0.4679334916864608
Recall: 0.8277310924369747
Cost: 470.0


Threshold: 0.25
Precision: 0.49736842105263157
Recall: 0.7941176470588235
Cost: 485.0


Threshold: 0.3
Precision: 0.5438066465256798
Recall: 0.7563025210084033
Cost: 499.0


Threshold: 0.35
Precision: 0.5694915254237288
Recall: 0.7058823529411765
Cost: 547.0


Threshold: 0.4
Precision: 0.5823754789272031
Recall: 0.6386554621848739
Cost: 625.0




In [ ]:
# Based on further exploration, a threshold of 0.3 gives the lowest cost
THRESHOLD = 0.3

More complex models for better performance

Random Forest

In [ ]:
# Random forest classifier
random_forest = RandomForestClassifier(random_state=42, class_weight='balanced')

start_train = time.time()
random_forest.fit(X_train, y_train)
print(f"Training time: {(time.time() - start_train):.2f} seconds")

Training time: 0.65 seconds


In [ ]:
# Make prediction and evaluate model
start_time = time.time()
y_pred = random_forest.predict_proba(X_val)[:,1] > THRESHOLD
print(f"Prediction time: {(time.time() - start_time):.2f} seconds")
print(f"Cost: {business_cost(y_val, y_pred)}\n")
print(classification_report(y_val, y_pred))

Prediction time: 0.02 seconds
Cost: 572.0

              precision    recall  f1-score   support

           0       0.88      0.78      0.83       660
           1       0.53      0.70      0.61       238

    accuracy                           0.76       898
   macro avg       0.71      0.74      0.72       898
weighted avg       0.79      0.76      0.77       898



At a threshold of 0.3, the random forest classifier seems to perform worse than the base model.

LightGBM

In [ ]:
lightgbm = LGBMClassifier(random_state=42, class_weight='balanced')

start_train = time.time()
lightgbm.fit(X_train, y_train)
print(f"Training time: {(time.time() - start_train):.2f} seconds")
print(f"\nModel Parameters: {lightgbm.get_params()}")

[LightGBM] [Info] Number of positive: 1351, number of negative: 3737
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000837 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 670
[LightGBM] [Info] Number of data points in the train set: 5088, number of used features: 46
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
Training time: 0.18 seconds

Model Parameters: {'boosting_type': 'gbdt', 'class_weight': 'balanced', 'colsample_bytree': 1.0, 'importance_type': 'split', 'learning_rate': 0.1, 'max_depth': -1, 'min_child_samples': 20, 'min_child_weight': 0.001, 'min_split_gain': 0.0, 'n_estimators': 100, 'n_jobs': None, 'num_leaves': 31, 'objective': None, 'random_state': 42, 'reg_alpha': 0.0, 'reg_lambda': 0.0, 'subsample': 1.0, 'subsample_for_bin': 200000, 's

In [ ]:
start_time = time.time()
y_pred = lightgbm.predict_proba(X_val)[:,1] > THRESHOLD
print(f"Prediction time: {(time.time() - start_time):.2f} seconds")
print(f"Cost: {business_cost(y_val, y_pred)}\n")
print(classification_report(y_val, y_pred))

Prediction time: 0.01 seconds
Cost: 450.0

              precision    recall  f1-score   support

           0       0.92      0.64      0.75       660
           1       0.46      0.85      0.60       238

    accuracy                           0.69       898
   macro avg       0.69      0.74      0.67       898
weighted avg       0.80      0.69      0.71       898



/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


The LightGBM model achieved a lower cost than the base model and the random forest classifier

XGBoost

In [ ]:
# Calculate scale_pos_weight for imbalance
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgboost = XGBClassifier(random_state=42, scale_pos_weight=scale_pos_weight)

start_train = time.time()
xgboost.fit(X_train, y_train)
print(f"Training time: {(time.time() - start_train):.2f} seconds")

Training time: 0.49 seconds


In [ ]:
start_time = time.time()
y_pred = xgboost.predict_proba(X_val)[:,1] > THRESHOLD
print(f"Prediction time: {(time.time() - start_time):.2f} seconds")
print(f"Cost: {business_cost(y_val, y_pred)}\n")
print(classification_report(y_val, y_pred))

Prediction time: 0.01 seconds
Cost: 522.0

              precision    recall  f1-score   support

           0       0.90      0.69      0.78       660
           1       0.48      0.78      0.59       238

    accuracy                           0.71       898
   macro avg       0.69      0.73      0.69       898
weighted avg       0.78      0.71      0.73       898



XGBoost achieved a cost of 522.

### Hyperparameter Tuning

LightGBM

In [ ]:
def objective(trial):
    # Suggest hyperparameter values
    params = {
        'objective': None,
        'metric': 'binary_logloss',
        'boosting_type': 'gbdt',
        'class_weight': 'balanced',
        'n_estimators': trial.suggest_int('n_estimators', 50, 200),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 15, 100),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.4, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.4, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        'verbose': -1
    }

    # Train and evaluate model
    model = LGBMClassifier(**params)
    model.fit(X_train, y_train)
    y_pred = model.predict_proba(X_val)[:,1] > THRESHOLD

    return business_cost(y_val, y_pred)

In [ ]:
# Create a study object and minimize the objective function
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=30)

print("\nBest trial score (Cost):", study.best_value)

[I 2026-09-09 21:21:13,481] A new study created in memory with name: no-name-0d405894-ce84-4921-ba97-d3d2fe8c2827
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2026-09-09 21:21:13,661] Trial 0 finished with value: 642.3076923076925 and parameters: {'n_estimators': 141, 'learning_rate': 0.02084972835803554, 'num_leaves': 100, 'max_depth': 3, 'feature_fraction': 0.9508959519893887, 'bagging_fraction': 0.7529958656416864, 'bagging_freq': 7, 'min_child_samples': 33}. Best is trial 0 with value: 642.3076923076925.
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2026-09-09 21:21:14,033] Trial 1 finished with value: 678.2125693160814 and parameters: {'n_estimators': 131, 'learning_rate': 0.0123161374050180


Best trial score (Cost): 422.0


In [ ]:
# Initialize model with the best parameters from the search
best_params = study.best_params
print(best_params)

model = LGBMClassifier(**best_params)
model.fit(X_train, y_train)
y_pred = model.predict_proba(X_val)[:,1] > THRESHOLD

print(f"Cost: {business_cost(y_val, y_pred)}\n")
print(classification_report(y_val, y_pred))

{'n_estimators': 97, 'learning_rate': 0.06503170843772528, 'num_leaves': 90, 'max_depth': 9, 'feature_fraction': 0.6961977804380988, 'bagging_fraction': 0.8448930051896469, 'bagging_freq': 3, 'min_child_samples': 65}
Cost: 537.0

              precision    recall  f1-score   support

           0       0.89      0.77      0.82       660
           1       0.53      0.73      0.62       238

    accuracy                           0.76       898
   macro avg       0.71      0.75      0.72       898
weighted avg       0.79      0.76      0.77       898



/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


XGBoost

In [ ]:
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 300, 800),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "gamma": trial.suggest_float("gamma", 0, 5),
        "reg_alpha": trial.suggest_float("reg_alpha", 0, 5),
        "reg_lambda": trial.suggest_float("reg_lambda", 0, 5),
        "random_state": 42,
        "n_jobs": -1,
        "scale_pos_weight": (y_train == 0).sum() / (y_train == 1).sum(),
        "eval_metric": "logloss"
    }

    model = XGBClassifier(**params)
    model.fit(X_train, y_train)
    proba = model.predict_proba(X_val)[:,1]
    y_pred = (proba >= THRESHOLD).astype(int)
    return business_cost(y_val, y_pred)

In [ ]:
# Run Optuna
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=30)

print("Best Params:", study.best_params)
print("Best Cost:", study.best_value)

[I 2026-09-09 21:26:08,552] A new study created in memory with name: no-name-160f126e-1565-42d8-b780-02f235eeed2a
[I 2026-09-09 21:26:08,996] Trial 0 finished with value: 473.51599147121533 and parameters: {'n_estimators': 536, 'learning_rate': 0.037018829892997526, 'max_depth': 7, 'subsample': 0.7021305326552456, 'colsample_bytree': 0.62350117309638, 'min_child_weight': 8, 'gamma': 2.2095630851453185, 'reg_alpha': 1.894588373049103, 'reg_lambda': 0.9907604211473647}. Best is trial 0 with value: 473.51599147121533.
[I 2026-09-09 21:26:09,419] Trial 1 finished with value: 484.932059447983 and parameters: {'n_estimators': 625, 'learning_rate': 0.028566987971537026, 'max_depth': 10, 'subsample': 0.6210877375376902, 'colsample_bytree': 0.7080957663575247, 'min_child_weight': 3, 'gamma': 4.034482440559756, 'reg_alpha': 3.783440626748929, 'reg_lambda': 3.09525407574956}. Best is trial 0 with value: 473.51599147121533.
[I 2026-09-09 21:26:09,892] Trial 2 finished with value: 480.0 and paramet

Best Params: {'n_estimators': 785, 'learning_rate': 0.018321864790821, 'max_depth': 9, 'subsample': 0.6823351704322228, 'colsample_bytree': 0.630517177462101, 'min_child_weight': 7, 'gamma': 1.9638909530154491, 'reg_alpha': 2.2621620798134314, 'reg_lambda': 1.3330117513884856}
Best Cost: 421.0


In [ ]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

# Add the scale_pos_weight and fixed params to the best ones from Optuna
best_params = study.best_params
best_params.update({
    "random_state": 42,
    "n_jobs": -1,
    "scale_pos_weight": scale_pos_weight,
    "eval_metric": "logloss"
})

# Create model from best params
xgb = XGBClassifier(**best_params)
xgb.fit(X_train, y_train)

proba = xgb.predict_proba(X_val)[:,1]
y_pred = (proba >= THRESHOLD).astype(int)

# Classification report
print(f"Cost: {business_cost(y_val, y_pred)}\n")
print(classification_report(y_val, y_pred, digits=3))

Cost: 421.0

              precision    recall  f1-score   support

           0      0.936     0.617     0.743       660
           1      0.454     0.882     0.599       238

    accuracy                          0.687       898
   macro avg      0.695     0.750     0.671       898
weighted avg      0.808     0.687     0.705       898



After tuning, XGBoost achieved the lowest cost: 421.
This makes XGBoost the model choice.

In [ ]:
# Threshold tuning for XGBoost
for threshold in [0.2, 0.25, 0.3, 0.35, 0.4]:
  y_pred = proba > threshold
  print(f"Threshold: {threshold}")
  print(f"Precision: {precision_score(y_val, y_pred, pos_label=1)}")
  print(f"Recall: {recall_score(y_val, y_pred, pos_label=1)}")
  print(f"Cost: {business_cost(y_val, y_pred)}")
  print("\n")

Threshold: 0.2
Precision: 0.40262172284644193
Recall: 0.9033613445378151
Cost: 693.8913857677904


Threshold: 0.25
Precision: 0.4306122448979592
Recall: 0.8865546218487395
Cost: 537.9387755102041


Threshold: 0.3
Precision: 0.4535637149028078
Recall: 0.8823529411764706
Cost: 421.0


Threshold: 0.35
Precision: 0.468384074941452
Recall: 0.8403361344537815
Cost: 455.0


Threshold: 0.4
Precision: 0.4849246231155779
Recall: 0.8109243697478992
Cost: 475.0




### Experiment Tracking

In [ ]:
# Force MLflow to always use the project root's mlruns folder
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
mlflow.set_tracking_uri(f"file://{project_root}/mlruns")
mlflow.set_experiment("Telco Churn - XGBoost")

with mlflow.start_run():
    # Calculate scale_pos_weight
    scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

    # Best params from Optuna
    best_params = study.best_params
    best_params.update({
        "random_state": 42,
        "n_jobs": -1,
        "scale_pos_weight": scale_pos_weight,
        "eval_metric": "logloss"
    })

    # Log parameters
    mlflow.log_params(best_params)

    xgb = XGBClassifier(**best_params)
    xgb.fit(X_train, y_train)

    proba = xgb.predict_proba(X_test)[:, 1]
    y_pred = (proba >= THRESHOLD).astype(int)

    precision = precision_score(y_test, y_pred, pos_label=1)
    recall = recall_score(y_test, y_pred, pos_label=1)
    cost = business_cost(y_test, y_pred)

    # Log metrics
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("cost", cost)

    # Save model
    mlflow.xgboost.log_model(xgb, "model")

    print(f"Cost: {cost}\n")
    print(classification_report(y_test, y_pred))

2026/09/09 21:53:28 INFO mlflow.tracking.fluent: Experiment with name 'Telco Churn - XGBoost' does not exist. Creating a new experiment.
2026/09/09 21:53:31 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Cost: 476.0

              precision    recall  f1-score   support

           0       0.94      0.62      0.75       777
           1       0.46      0.89      0.61       280

    accuracy                           0.69      1057
   macro avg       0.70      0.76      0.68      1057
weighted avg       0.81      0.69      0.71      1057

